# Point Cloud · CPU vs GPU 실습
**런타임 → 런타임 유형 변경 → GPU** 선택 후 아래 **① 준비 → ② 뷰어** 두 셀을 실행하세요.

- Bunny / Dragon / Happy Buddha / Drill을 최대 **64×64**로 배열합니다.
- 공전·접근/후퇴·나선·내부 비행·호버링: 경로 선택 후 **경로 재생**.
- 화면 드래그=각도, 휠=거리.
- **K 프레임 렌더링**: K 입력, 비교 Both 선택. 같은 K개 카메라 자세를 CPU/GPU로 계산하고 프레임별 ms·총시간·평균·중앙값·p95를 출력합니다.
- 이미지 전송이 느려도 K 프레임 측정은 서버 안에서 진행합니다. GPU 측정은 pose 업로드·계산·depth CPU 회수를 포함하며, 이미지 변환·전송·표시는 제외합니다.


In [ ]:
#@title ① 준비 — 패키지와 실습 파일
import importlib.util, subprocess, sys
packages={'numpy':'numpy','numba':'numba','PIL':'pillow','matplotlib':'matplotlib','plyfile':'plyfile','cupy':'cupy-cuda12x[ctk]'}
missing=[pkg for module,pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','-q','install',*missing])
from pathlib import Path
Path('bunny_depth_lab.py').write_text('"""RT616: Stanford Bunny camera-depth rasterization, CPU and CUDA.\nData: https://graphics.stanford.edu/data/3Dscanrep/ (bunny/reconstruction/bun_zipper.ply)\nOne point -> one pixel; optical-axis z in scene units; not ray tracing or LiDAR range.\nCPU: compiled Numba single-thread fused loop. GPU: fused CuPy RawKernel.\n"""\nimport os, sys, json, time, tarfile, urllib.request, platform, hashlib\nfrom pathlib import Path\nimport numpy as np\nfrom numba import njit\n\n# Windows pip CUDA DLLs; no change to the system driver or global PATH.\n_dll_handles=[]\nif os.name == \'nt\':\n    for root in [Path(sys.prefix)/\'Lib/site-packages/nvidia\']:\n        if root.exists():\n            for p in root.glob(\'*/bin\'):\n                _dll_handles.append(os.add_dll_directory(str(p)))\nGPU_ERROR=""\ntry:\n    if os.environ.get("RT616_CPU_ONLY") == "1":\n        raise RuntimeError("CPU-only mode requested")\n    import cupy as cp\n    GPU_AVAILABLE=cp.cuda.runtime.getDeviceCount()>0\nexcept Exception as exc:\n    cp=None; GPU_AVAILABLE=False; GPU_ERROR=str(exc)\n\nCUDA_SOURCE=r\'\'\'\nextern "C" __global__ void project_depth(\n    const float* xyz, const float* R, const float* eye,\n    unsigned int* depth, int n, int w, int h, float f) {\n    int i=blockIdx.x*blockDim.x+threadIdx.x;\n    if(i>=n) return;\n    float x=xyz[3*i]-eye[0], y=xyz[3*i+1]-eye[1], z=xyz[3*i+2]-eye[2];\n    float X=R[0]*x+R[1]*y+R[2]*z;\n    float Y=R[3]*x+R[4]*y+R[5]*z;\n    float Z=R[6]*x+R[7]*y+R[8]*z;\n    if(Z<=0.01f) return;\n    int u=(int)floorf(f*X/Z+0.5f*w);\n    int v=(int)floorf(0.5f*h-f*Y/Z);\n    if(u>=0 && u<w && v>=0 && v<h)\n        atomicMin(depth+v*w+u,__float_as_uint(Z));\n}\n\'\'\'\n\ndef load_bunny(cache=\'rt616_data\'):\n    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    archive=cache/\'bunny.tar.gz\'\n    if not archive.exists():\n        bundled=(Path(__file__).resolve().parent if \'__file__\' in globals() else Path.cwd())/\'bunny.tar.gz\'\n        if bundled.exists():\n            import shutil\n            shutil.copyfile(bundled,archive)\n        else:\n            urllib.request.urlretrieve(\'https://graphics.stanford.edu/pub/3Dscanrep/bunny.tar.gz\',archive)\n    # Read one known member; never extract archive paths.\n    with tarfile.open(archive,\'r:gz\') as tf:\n        raw=tf.extractfile(\'bunny/reconstruction/bun_zipper.ply\').read()\n    header, body=raw.split(b\'end_header\\n\',1)\n    assert b\'format ascii\' in header\n    n=int(next(x for x in header.decode().splitlines() if x.startswith(\'element vertex\')).split()[-1])\n    pts=np.asarray([[float(v) for v in line.split()[:3]] for line in body.decode().splitlines()[:n]],np.float32)\n    pts-=np.array([(pts[:,0].min()+pts[:,0].max())/2,pts[:,1].min(),(pts[:,2].min()+pts[:,2].max())/2],np.float32)\n    pts/=np.ptp(pts[:,1])\n    return pts, hashlib.sha256(archive.read_bytes()).hexdigest()\n\ndef make_scene(bunny,side=8):\n    offsets=np.array([[(i-(side-1)/2)*1.5,0,(j-(side-1)/2)*1.5] for i in range(side) for j in range(side)],np.float32)\n    return np.ascontiguousarray((bunny[None,:,:]+offsets[:,None,:]).reshape(-1,3))\n\ndef camera(angle,side=8):\n    radius=side*1.15\n    eye=np.array([radius*np.sin(angle),side*.7,radius*np.cos(angle)],np.float32)\n    forward=np.array([0,.3,0],np.float32)-eye; forward/=np.linalg.norm(forward)\n    right=np.cross(forward,np.array([0,1,0],np.float32)); right/=np.linalg.norm(right)\n    up=np.cross(right,forward)\n    return np.ascontiguousarray(np.stack([right,up,forward])),eye\n\n@njit(cache=True,fastmath=False)\ndef render_cpu(points,R,eye,w=640,h=480,f=np.float32(500)):\n    depth=np.full((h,w),np.float32(np.inf),np.float32)\n    for i in range(len(points)):\n        x=points[i,0]-eye[0]; y=points[i,1]-eye[1]; z=points[i,2]-eye[2]\n        X=R[0,0]*x+R[0,1]*y+R[0,2]*z\n        Y=R[1,0]*x+R[1,1]*y+R[1,2]*z\n        Z=R[2,0]*x+R[2,1]*y+R[2,2]*z\n        if Z<=np.float32(.01): continue\n        u=int(np.floor(f*X/Z+np.float32(.5*w)))\n        v=int(np.floor(np.float32(.5*h)-f*Y/Z))\n        if 0<=u<w and 0<=v<h and Z<depth[v,u]: depth[v,u]=Z\n    return depth\n\ndef render_numpy(points,R,eye,w=640,h=480,f=np.float32(500)):\n    # Independent vectorized reference; not the optimized CPU timing baseline.\n    p=(points-eye)@R.T\n    valid=p[:,2]>np.float32(.01); p=p[valid]\n    u=np.floor(f*p[:,0]/p[:,2]+np.float32(.5*w)).astype(np.int32)\n    v=np.floor(np.float32(.5*h)-f*p[:,1]/p[:,2]).astype(np.int32)\n    mask=(u>=0)&(u<w)&(v>=0)&(v<h)\n    out=np.full(w*h,np.inf,np.float32)\n    np.minimum.at(out,v[mask]*w+u[mask],p[mask,2])\n    return out.reshape(h,w)\n\nclass GPURenderer:\n    def __init__(self,points,w=640,h=480):\n        if not GPU_AVAILABLE: raise RuntimeError(\'CUDA GPU unavailable; select a GPU runtime in Colab\')\n        self.points=cp.asarray(points); self.w=w; self.h=h\n        self.depth=cp.empty((h,w),cp.float32)\n        self.R=cp.empty((3,3),cp.float32); self.eye=cp.empty(3,cp.float32)\n        self.kernel=cp.RawKernel(CUDA_SOURCE,\'project_depth\',options=(\'--fmad=false\',))\n        cp.cuda.get_current_stream().synchronize()\n    def set_camera(self,R,eye): self.R.set(R); self.eye.set(eye)\n    def compute(self):\n        self.depth.fill(cp.inf)\n        if len(self.points)==0: return self.depth\n        self.kernel(((len(self.points)+255)//256,),(256,),\n            (self.points,self.R,self.eye,self.depth,np.int32(len(self.points)),np.int32(self.w),np.int32(self.h),np.float32(500)))\n        return self.depth\n    def frame(self,R,eye,copy_points=None):\n        if copy_points is not None: self.points.set(copy_points)\n        self.set_camera(R,eye); self.compute()\n        return cp.asnumpy(self.depth)\n\ndef stats(times):\n    a=np.array(times)*1000\n    return dict(median_ms=float(np.median(a)),p95_ms=float(np.percentile(a,95)),iqr_ms=float(np.percentile(a,75)-np.percentile(a,25)))\n\ndef benchmark(side=8,repeats=30,w=640,h=480,cache=\'rt616_data\'):\n    bunny,sha=load_bunny(cache); points=make_scene(bunny,side); R,eye=camera(.2,side)\n    cpu=render_cpu(points,R,eye,w,h) # JIT cold time excluded\n    info=dict(cpu=platform.processor(),platform=platform.platform(),python=sys.version.split()[0],\n              numpy=np.__version__,side=side,bunnies=side*side,points=len(points),width=w,height=h,\n              repeats=repeats,dataset_sha256=sha,cpu_baseline=\'Numba fused single-thread; fastmath=False\',\n              scope=\'warm compute; GPU resident frame includes camera upload and depth download; per-frame point-upload measured separately; browser display excluded\')\n    times=[]\n    for _ in range(repeats):\n        t=time.perf_counter(); render_cpu(points,R,eye,w,h); times.append(time.perf_counter()-t)\n    info[\'cpu\']=dict(name=platform.processor(),**stats(times))\n    if not GPU_AVAILABLE:\n        info[\'gpu_unavailable\']=globals().get(\'GPU_ERROR\',\'No device\'); return info\n    renderer=GPURenderer(points,w,h); gpu=renderer.frame(R,eye)\n    props=cp.cuda.runtime.getDeviceProperties(0)\n    info[\'device\']=dict(name=props[\'name\'].decode(),compute_capability=f"{props[\'major\']}.{props[\'minor\']}",\n                        total_bytes=int(props[\'totalGlobalMem\']),cupy=cp.__version__,runtime=cp.cuda.runtime.runtimeGetVersion(),driver=cp.cuda.runtime.driverGetVersion())\n    common=np.isfinite(cpu)&np.isfinite(gpu); union=np.isfinite(cpu)|np.isfinite(gpu)\n    info[\'accuracy\']=dict(mask_iou=float(common.sum()/max(1,union.sum())),max_depth_error=float(np.max(np.abs(cpu[common]-gpu[common]))),\n                          mean_depth_error=float(np.mean(np.abs(cpu[common]-gpu[common]))))\n    assert info[\'accuracy\'][\'mask_iou\']>.999,info[\'accuracy\']\n    assert info[\'accuracy\'][\'mean_depth_error\']<1e-4,info[\'accuracy\']\n    times=[]\n    for _ in range(repeats):\n        start,end=cp.cuda.Event(),cp.cuda.Event(); start.record(); renderer.compute(); end.record(); end.synchronize()\n        times.append(cp.cuda.get_elapsed_time(start,end)/1000)\n    info[\'gpu_compute\']=stats(times)\n    for label,upload in [(\'gpu_resident_frame\',None),(\'gpu_upload_each_frame\',points)]:\n        times=[]\n        for _ in range(repeats):\n            t=time.perf_counter(); renderer.frame(R,eye,upload); times.append(time.perf_counter()-t)\n        info[label]=stats(times)\n    info[\'speedup_resident\']=info[\'cpu\'][\'median_ms\']/info[\'gpu_resident_frame\'][\'median_ms\']\n    return info\n\ndef depth_rgb(depth,near=0,far=24):\n    # A visualization of measured optical-axis depth, with fixed limits across frames.\n    from matplotlib import colormaps\n    valid=np.isfinite(depth); a=np.zeros_like(depth)\n    a[valid]=np.clip((depth[valid]-near)/(far-near),0,1)\n    rgb=(colormaps[\'turbo\'](a)[...,:3]*255).astype(np.uint8); rgb[~valid]=245\n    return rgb\n\ndef interactive_view(side=8,backend=\'GPU\'):\n    import ipywidgets as widgets\n    from IPython.display import display,clear_output\n    from PIL import Image\n    bunny,_=load_bunny(); points=make_scene(bunny,side)\n    renderer=GPURenderer(points) if GPU_AVAILABLE else None\n    render_cpu(points,*camera(0,side))\n    angle=widgets.FloatSlider(min=-180,max=180,value=0,step=3,description=\'Yaw\',continuous_update=False)\n    choice=widgets.Dropdown(options=[\'CPU\']+([\'GPU\'] if renderer else []),value=backend if renderer else \'CPU\',description=\'Backend\')\n    out=widgets.Output()\n    def draw(_=None):\n        R,eye=camera(np.deg2rad(angle.value),side)\n        t=time.perf_counter(); depth=renderer.frame(R,eye) if choice.value==\'GPU\' else render_cpu(points,R,eye)\n        ms=(time.perf_counter()-t)*1000\n        with out:\n            clear_output(wait=True)\n            print(f\'{len(points):,} points | {choice.value} frame compute/copy {ms:.2f} ms | notebook display excluded\')\n            display(Image.fromarray(depth_rgb(depth,far=side*3)))\n    angle.observe(draw,names=\'value\'); choice.observe(draw,names=\'value\'); display(widgets.VBox([angle,choice,out])); draw()\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser(); p.add_argument(\'--out\',default=\'results\'); p.add_argument(\'--repeats\',type=int,default=30)\n    args=p.parse_args(); out=Path(args.out); out.mkdir(parents=True,exist_ok=True)\n    results=[benchmark(side=s,repeats=args.repeats,cache=out/\'data\') for s in [2,8,12]]\n    (out/\'benchmark.json\').write_text(json.dumps(results,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    print(json.dumps(results,indent=2,ensure_ascii=False))\n    from PIL import Image\n    bunny,_=load_bunny(out/\'data\'); points=make_scene(bunny,8); renderer=GPURenderer(points) if GPU_AVAILABLE else None\n    for j,a in enumerate([0,.55,1.1]):\n        R,eye=camera(a,8); d=renderer.frame(R,eye) if renderer else render_cpu(points,R,eye)\n        Image.fromarray(depth_rgb(d,far=24)).save(out/f\'bunny-depth-{j}.png\')\n',encoding='utf-8')
Path('stanford_scene.py').write_text('"""Instanced Stanford models, camera trajectories, local and Colab controls."""\nfrom bunny_depth_lab import *\nimport io,gc\nMODELS={\n \'Bunny\':(\'https://graphics.stanford.edu/pub/3Dscanrep/bunny.tar.gz\',\'bun_zipper.ply\'),\n \'Dragon\':(\'https://graphics.stanford.edu/pub/3Dscanrep/dragon/dragon_recon.tar.gz\',\'dragon_vrip.ply\'),\n \'Happy Buddha\':(\'https://graphics.stanford.edu/pub/3Dscanrep/happy/happy_recon.tar.gz\',\'happy_vrip.ply\'),\n \'Drill\':(\'https://graphics.stanford.edu/pub/3Dscanrep/drill.tar.gz\',\'drill_shaft_vrip.ply\')}\nPATHS=[\'Orbit\',\'Dolly\',\'Helix\',\'Fly-through\',\'Hover\']\ndef load_model(name=\'Bunny\',cap=36000,cache=\'rt616_data\'):\n    from plyfile import PlyData\n    url,suffix=MODELS[name];folder=Path(cache);folder.mkdir(parents=True,exist_ok=True)\n    archive=folder/url.split(\'/\')[-1]\n    if not archive.exists():urllib.request.urlretrieve(url,archive)\n    with tarfile.open(archive,\'r:gz\') as tf:\n        candidates=[m for m in tf.getmembers() if m.name.endswith(suffix)]\n        if not candidates:raise ValueError(f\'{name}: missing {suffix}; PLY members: {[m.name for m in tf.getmembers() if m.name.endswith(".ply")]}\')\n        ply=PlyData.read(io.BytesIO(tf.extractfile(candidates[0]).read()))\n    v=ply[\'vertex\'];pts=np.column_stack([v[k] for k in [\'x\',\'y\',\'z\']]).astype(np.float32)\n    count=len(pts);pts-=np.array([(pts[:,0].min()+pts[:,0].max())/2,pts[:,1].min(),(pts[:,2].min()+pts[:,2].max())/2],np.float32)\n    pts/=np.ptp(pts[:,1])\n    if cap and count>cap:\n        pts=pts[np.sort(np.random.default_rng(616).choice(count,int(cap),replace=False))]\n    return np.ascontiguousarray(pts),dict(model=name,source=url,original_points=count,points_per_instance=len(pts),sample_seed=616,sha256=hashlib.sha256(archive.read_bytes()).hexdigest())\n\ndef pose(t,side=8,path=\'Orbit\',yaw=0,pitch=25,distance=1.2):\n    """t is normalized path phase; distance is relative to the grid extent."""\n    a=2*np.pi*t+np.deg2rad(yaw);r=side*distance;target=np.array([0,.5,0],np.float32)\n    elev=np.deg2rad(pitch)\n    if path==\'Dolly\':r*=.2+.8*(.5+.5*np.cos(2*np.pi*t));a=np.deg2rad(yaw)\n    elif path==\'Helix\':elev=np.deg2rad(np.clip(pitch+32.5*np.sin(2*np.pi*t),-75,85))\n    elif path==\'Hover\':r*=1+.15*np.sin(6*np.pi*t);elev+=.15*np.sin(4*np.pi*t)\n    if path==\'Fly-through\':\n        x=.55*np.sin(2*np.pi*t);z=side*distance*np.cos(2*np.pi*t);y=np.deg2rad(yaw)\n        eye=np.array([x*np.cos(y)+z*np.sin(y),1.1+.5*np.sin(4*np.pi*t)+(pitch-25)*.02,-x*np.sin(y)+z*np.cos(y)],np.float32)\n        target=np.array([0,.5,0],np.float32)\n    else:eye=target+np.array([r*np.cos(elev)*np.sin(a),r*np.sin(elev),r*np.cos(elev)*np.cos(a)],np.float32)\n    f=target-eye;f/=np.linalg.norm(f);right=np.cross(f,np.array([0,1,0],np.float32));right/=np.linalg.norm(right)\n    up=np.cross(right,f)\n    return np.ascontiguousarray(np.stack([right,up,f]),np.float32),np.ascontiguousarray(eye,np.float32)\n\n@njit(cache=True,fastmath=False)\ndef render_instances_cpu(base,offsets,R,eye,w=640,h=480,f=np.float32(500)):\n    d=np.full((h,w),np.float32(np.inf),np.float32)\n    for j in range(len(offsets)):\n        for i in range(len(base)):\n            x=(base[i,0]+offsets[j,0])-eye[0];y=(base[i,1]+offsets[j,1])-eye[1];z=(base[i,2]+offsets[j,2])-eye[2]\n            X=R[0,0]*x+R[0,1]*y+R[0,2]*z;Y=R[1,0]*x+R[1,1]*y+R[1,2]*z;Z=R[2,0]*x+R[2,1]*y+R[2,2]*z\n            if Z<=np.float32(.01):continue\n            u=int(np.floor(f*X/Z+np.float32(w*.5)));v=int(np.floor(np.float32(h*.5)-f*Y/Z))\n            if 0<=u<w and 0<=v<h and Z<d[v,u]:d[v,u]=Z\n    return d\n\nINSTANCE_KERNEL=CUDA_SOURCE.replace(\'const float* xyz,\',\'const float* xyz, const float* offsets, int base_n,\').replace(\n \'float x=xyz[3*i]-eye[0], y=xyz[3*i+1]-eye[1], z=xyz[3*i+2]-eye[2];\',\n \'int b=i%base_n, j=i/base_n; float x=(xyz[3*b]+offsets[3*j])-eye[0], y=(xyz[3*b+1]+offsets[3*j+1])-eye[1], z=(xyz[3*b+2]+offsets[3*j+2])-eye[2];\')\nclass Scene:\n    def __init__(self,model=\'Bunny\',side=8,cap=36000,cache=\'rt616_data\'):\n        if side not in [2,8,12,16,32,64]:raise ValueError(\'Unsupported grid\')\n        self.base,self.info=load_model(model,cap,cache);self.side=side\n        spacing=max(1.5,float(np.ptp(self.base[:,0]))*1.15,float(np.ptp(self.base[:,2]))*1.15)\n        self.offsets=np.array([[(i-(side-1)/2)*spacing,0,(j-(side-1)/2)*spacing] for i in range(side) for j in range(side)],np.float32)\n        self.extent=side*spacing/1.5\n        self.n=len(self.base)*len(self.offsets)\n        if self.n>=2**31:raise ValueError(\'Too many points: reduce points/model\')\n        self.info.update(grid=side,instances=side*side,logical_points=self.n,stored_bytes=self.base.nbytes+self.offsets.nbytes,spacing=spacing)\n        if GPU_AVAILABLE:\n            self.bg=cp.asarray(self.base);self.og=cp.asarray(self.offsets);self.Rg=cp.empty((3,3),cp.float32);self.eg=cp.empty(3,cp.float32);self.dg=cp.empty((480,640),cp.float32)\n            self.kernel=cp.RawKernel(INSTANCE_KERNEL,\'project_depth\',options=(\'--fmad=false\',));self.frame(*pose(0,self.extent),\'GPU\')\n        render_instances_cpu(self.base[:1],self.offsets[:1],*pose(0,self.extent))\n    def frame(self,R,e,backend=\'GPU\',upload=False):\n        if backend==\'CPU\':return render_instances_cpu(self.base,self.offsets,R,e)\n        if not GPU_AVAILABLE:raise RuntimeError(\'GPU unavailable; select a Colab GPU runtime or CPU\')\n        if upload:self.bg.set(self.base);self.og.set(self.offsets)\n        self.Rg.set(R);self.eg.set(e);self.dg.fill(cp.inf)\n        self.kernel(((self.n+255)//256,),(256,),(self.bg,self.og,np.int32(len(self.base)),self.Rg,self.eg,self.dg,np.int32(self.n),np.int32(640),np.int32(480),np.float32(500)))\n        return cp.asnumpy(self.dg)\n\ndef trajectory(scene,path=\'Hover\',frames=60,backend=\'GPU\',yaw=0,pitch=25,distance=1.2,out=\'trajectory\'):\n    """Export a rendered GIF plus actual per-frame backend times and camera extrinsics."""\n    from PIL import Image\n    images=[];rows=[]\n    for i in range(frames):\n        t=i/frames;R,e=pose(t,scene.extent,path,yaw,pitch,distance)\n        start=time.perf_counter();d=scene.frame(R,e,backend);ms=(time.perf_counter()-start)*1000\n        images.append(Image.fromarray(depth_rgb(d,far=scene.extent*3)))\n        rows.append(dict(frame=i,phase=t,ms=ms,eye=e.tolist(),R_world_to_camera=R.tolist()))\n    images[0].save(str(out)+\'.gif\',save_all=True,append_images=images[1:],duration=50,loop=0)\n    Path(str(out)+\'.json\').write_text(json.dumps(dict(scene=scene.info,path=path,backend=backend,frames=rows,playback=\'Fixed 20 fps preview, NOT measured throughput\'),indent=2),encoding=\'utf-8\')\n    return rows\n\ndef colab_viewer():\n    import ipywidgets as w\n    from IPython.display import display,clear_output\n    from PIL import Image\n    model=w.Dropdown(options=list(MODELS),description=\'Model\');side=w.Dropdown(options=[2,8,12,16,32,64],value=8,description=\'Grid\')\n    cap=w.Dropdown(options=[12000,36000,100000],value=36000,description=\'Points/model\')\n    path=w.Dropdown(options=PATHS,value=\'Hover\',description=\'Path\');backend=w.Dropdown(options=[\'GPU\',\'CPU\'] if GPU_AVAILABLE else [\'CPU\'],description=\'Backend\')\n    yaw=w.FloatSlider(min=-180,max=180,description=\'Yaw\',continuous_update=False);pitch=w.FloatSlider(min=-60,max=85,value=25,description=\'Pitch\',continuous_update=False)\n    distance=w.FloatSlider(min=.1,max=2.5,value=1.2,step=.05,description=\'Distance/Z\',continuous_update=False)\n    phase=w.IntSlider(min=0,max=119,description=\'Path phase\',continuous_update=False);play=w.Play(min=0,max=119,interval=100);w.jslink((play,\'value\'),(phase,\'value\'))\n    build=w.Button(description=\'Load scene\');export=w.Button(description=\'Render 60-frame GIF\');output=w.Output();state={}\n    def draw(_=None):\n        if \'scene\' not in state:return\n        s=state[\'scene\'];R,e=pose(phase.value/120,s.extent,path.value,yaw.value,pitch.value,distance.value)\n        start=time.perf_counter();d=s.frame(R,e,backend.value);ms=(time.perf_counter()-start)*1000\n        with output:\n            clear_output(wait=True);print(s.info);print(f\'{backend.value}: {ms:.2f} ms; widget/display excluded; eye={e}\')\n            display(Image.fromarray(depth_rgb(d,far=s.extent*3)))\n    def load(_):\n        play.value=0\n        with output:clear_output(wait=True);print(\'Loading scene...\')\n        state.clear();gc.collect()\n        if GPU_AVAILABLE:cp.get_default_memory_pool().free_all_blocks()\n        state[\'scene\']=Scene(model.value,side.value,cap.value);draw()\n    def save(_):\n        if \'scene\' not in state:return\n        with output:\n            print(\'Rendering actual frames...\')\n            rows=trajectory(state[\'scene\'],path.value,60,backend.value,yaw.value,pitch.value,distance.value)\n            print(\'Saved trajectory.gif + trajectory.json; median backend ms:\',np.median([x[\'ms\'] for x in rows]))\n            display(__import__(\'IPython\').display.Image(filename=\'trajectory.gif\'))\n    build.on_click(load);export.on_click(save)\n    for widget in [path,backend,yaw,pitch,distance,phase]:widget.observe(draw,names=\'value\')\n    display(w.VBox([w.HBox([model,side,cap]),build,w.HBox([path,backend]),yaw,pitch,distance,w.HBox([play,phase]),export,output]));load(None)\n',encoding='utf-8')
Path('serve_demo.py').write_text('from stanford_scene import *\nfrom http.server import HTTPServer,BaseHTTPRequestHandler\nfrom urllib.parse import urlparse,parse_qs\nimport socket\nfrom benchmark_lab import benchmark_frames\nactive={}\ndef scene(model,side,cap):\n    key=(model,side,cap)\n    if active.get(\'key\')!=key:\n        active.clear();gc.collect()\n        if GPU_AVAILABLE:cp.get_default_memory_pool().free_all_blocks()\n        active[\'scene\']=Scene(model,side,cap);active[\'key\']=key\n    return active[\'scene\']\nHTML=r"""<!doctype html><meta charset="utf-8"><title>RT616 · Stanford Trajectory Lab</title>\n<style>body{font:16px Arial;margin:24px;background:#f7fafb;color:#183039}main{max-width:1150px}canvas{width:min(900px,100%);box-sizing:border-box;border:1px solid #ddd}button,select{font:inherit;margin:5px;padding:7px}label{display:inline-block;margin:5px}input{vertical-align:middle}small{display:block;line-height:1.6;max-width:950px}#status{font:14px monospace;margin:12px 0;min-height:36px}h1{margin-bottom:8px}</style>\n<main><h1>Stanford Trajectory Lab</h1><p>모델과 격자를 선택하고, 카메라가 접근·후퇴하거나 장면 사이를 비행하는 깊이 영상을 보세요.</p>\n<label>Model <select id="model"><option>Bunny</option><option>Dragon</option><option>Happy Buddha</option><option>Drill</option></select></label>\n<label>Grid <select id="side"><option>2</option><option selected>8</option><option>12</option><option>16</option><option>32</option><option>64</option></select></label>\n<label>Points/model <select id="cap"><option>12000</option><option selected>36000</option><option>100000</option></select></label>\n<label>Backend <select id="backend"><option>GPU</option><option>CPU</option></select></label><br>\n<label>Path <select id="path"><option>Orbit</option><option>Dolly</option><option>Helix</option><option>Fly-through</option><option selected>Hover</option></select></label>\n<button id="play">경로 재생</button><button id="reset">시점 초기화</button>\n<label><input id="upload" type="checkbox">GPU 원본+배치 재업로드</label><br>\n<label>Yaw <input id="yaw" type="range" min="-180" max="180" value="0"></label>\n<label>Pitch <input id="pitch" type="range" min="-60" max="85" value="25"></label>\n<label>Distance/Z <input id="distance" type="range" min="0.1" max="2.5" step="0.02" value="1.2"></label><br>\n<label>경로 위치 <input id="phase" type="range" min="0" max="1" step="0.002" value="0"></label>\n<label>속도 <input id="speed" type="range" min="0.1" max="3" step="0.1" value="1"></label>\n<section style="margin:14px 0;padding:12px;background:#e9f0f2">\n<label>K <input id="batchK" type="number" min="1" max="300" value="60" style="width:65px"></label>\n<label>비교 <select id="batchMode"><option>Both</option><option>CPU</option><option>GPU</option></select></label>\n<button id="batchRun">K 프레임 렌더링</button><button id="batchStop" disabled>중단</button>\n<small>현재 장면·경로를 한 바퀴 도는 동일한 K개 시점을 계산합니다. 이미지 전송 없이 서버 안에서 연속 측정합니다. Warmup 3회는 제외합니다.</small>\n<div id="batchSummary" style="font-weight:bold;margin:8px 0" aria-live="polite"></div>\n<pre id="batchLog" style="max-height:260px;overflow:auto;background:#122731;color:#bce6d1;padding:10px;font:13px monospace;white-space:pre-wrap">프레임별 CPU/GPU ms가 여기에 출력됩니다.</pre>\n<button id="batchSave" disabled>측정 JSON 저장</button>\n</section><div id="status">초기화 중… 처음 고르는 모델은 Stanford에서 다운로드합니다.</div><canvas id="view" width="640" height="480"></canvas>\n<small>Orbit: 공전 · Dolly: 접근/후퇴 · Helix: 고도가 변하는 공전 · Fly-through: 격자 내부 통과 · Hover: 거리·높이 변화.<br>\n원본 모델 + 배치 좌표만 저장하는 instancing을 CPU/GPU에 동일하게 적용합니다. 모든 표시 점을 처리하며 culling으로 점 수를 줄이지 않습니다. Points/model은 명시적인 결정적 샘플링 상한입니다. 재업로드는 이 압축 표현의 복사이며 이전 실습의 전체 점 배열 복사와 범위가 다릅니다.<br>\n깊이는 camera Z, 단위는 모델 높이=1인 scene unit. 가까운 곳은 청색, 먼 곳은 황색/적색이며 빈 픽셀은 연회색입니다. CPU는 컴파일된 단일 스레드입니다. 화면 표시·색칠·통신은 backend 시간에서 제외합니다.<br>\n<a href="https://colab.research.google.com/github/team-aprl/lecture-RT616-public/blob/main/practice/gpu/bunnys_projections/PointCloud_CPU_GPU.ipynb">Colab에서 같은 실험</a> · <a href="https://graphics.stanford.edu/data/3Dscanrep/">Stanford Computer Graphics Laboratory</a></small></main>\n<script>const $=x=>document.getElementById(x),ctx=$(\'view\').getContext(\'2d\');if(!__GPU_AVAILABLE__){\n $(\'backend\').value=\'CPU\';$(\'batchMode\').value=\'CPU\';\n for(const id of [\'backend\',\'batchMode\'])for(const option of $(id).options)if(option.value!==\'CPU\')option.disabled=true;\n}\nlet running=false,busy=false,pending=false,batching=false,batchAbort=null,batchRows=[],last=performance.now();\nasync function frame(){if(batching)return;if(busy){pending=true;return;}busy=true;pending=false;let begin=performance.now();\nconst p=new URLSearchParams();[\'model\',\'side\',\'cap\',\'backend\',\'path\',\'yaw\',\'pitch\',\'distance\',\'phase\'].forEach(k=>p.set(k,$(k).value));p.set(\'upload\',$(\'upload\').checked?\'1\':\'0\');p.set(\'format\',new URLSearchParams(location.search).get(\'transport\')===\'png\'?\'png\':\'raw\');\ntry{let r=await fetch(\'frame?\'+p);if(!r.ok)throw Error(await r.text());if(p.get(\'format\')===\'png\'){let bitmap=await createImageBitmap(await r.blob());ctx.drawImage(bitmap,0,0);bitmap.close();}else{let bytes=await r.arrayBuffer();ctx.putImageData(new ImageData(new Uint8ClampedArray(bytes),640,480),0,0);}\n$(\'status\').textContent=`${p.get(\'model\')} | ${r.headers.get(\'X-Points\')} points / ${p.get(\'side\')}×${p.get(\'side\')} instances | stored ${r.headers.get(\'X-Stored-MiB\')} MiB | ${p.get(\'backend\')} ${r.headers.get(\'X-Render-Ms\')} ms | viewer ${(performance.now()-begin).toFixed(1)} ms | eye ${r.headers.get(\'X-Eye\')}`;\n}catch(e){$(\'status\').textContent=e.message;running=false;$(\'play\').textContent=\'경로 재생\';}finally{busy=false;}\nif(running){let now=performance.now(),dt=Math.min((now-last)/1000,.25);last=now;$(\'phase\').value=(Number($(\'phase\').value)+dt*Number($(\'speed\').value)/20)%1;setTimeout(frame,Math.max(0,16.7-(performance.now()-begin)));}else if(pending)frame();}\n$(\'play\').onclick=()=>{running=!running;last=performance.now();$(\'play\').textContent=running?\'정지\':\'경로 재생\';if(running)frame();};\n$(\'reset\').onclick=()=>{$(\'yaw\').value=0;$(\'pitch\').value=25;$(\'distance\').value=1.2;$(\'phase\').value=0;frame();};\n[\'model\',\'side\',\'cap\',\'backend\',\'path\',\'yaw\',\'pitch\',\'distance\',\'phase\',\'upload\'].forEach(k=>$(k).onchange=frame);\nlet drag=null;$(\'view\').onpointerdown=e=>{drag=[e.clientX,e.clientY];$(\'view\').setPointerCapture(e.pointerId)};\n$(\'view\').onpointermove=e=>{if(!drag)return;$(\'yaw\').value=Math.max(-180,Math.min(180,Number($(\'yaw\').value)+(e.clientX-drag[0])*.3));$(\'pitch\').value=Math.max(-60,Math.min(85,Number($(\'pitch\').value)-(e.clientY-drag[1])*.2));drag=[e.clientX,e.clientY];frame()};$(\'view\').onpointerup=()=>drag=null;\n$(\'view\').onwheel=e=>{e.preventDefault();$(\'distance\').value=Math.max(.1,Math.min(2.5,Number($(\'distance\').value)+e.deltaY*.001));frame()};\nfunction batchLine(t){$(\'batchLog\').textContent+=t+\'\\n\';$(\'batchLog\').scrollTop=$(\'batchLog\').scrollHeight;}\nfunction batchEvent(e){\n batchRows.push(e);\n if(e.type===\'start\')batchLine(e.scene.model+\' | \'+e.scene.grid+\'×\'+e.scene.grid+\' | \'+e.scene.logical_points.toLocaleString()+\' points/frame | K=\'+e.k+\' | \'+e.path);\n if(e.type===\'warmup\')batchLine(e.message);\n if(e.type===\'frame\'){\n  batchLine(\'[\'+String(e.frame).padStart(3,\'0\')+\'/\'+e.k+\'] \'+[\'CPU\',\'GPU\'].filter(b=>b in e.ms).map(b=>b+\' \'+e.ms[b].toFixed(3)+\' ms\').join(\' | \')+(e.matching===undefined?\'\':\' | depth \'+(e.matching?\'MATCH\':\'MISMATCH\')));\n  $(\'batchSummary\').textContent=e.frame+\' / \'+e.k+\' 프레임 완료\';\n }\n if(e.type===\'done\'){\n  let lines=Object.entries(e.stats).map(([b,v])=>b+\': 총 \'+v.total_ms.toFixed(2)+\' ms | 평균 \'+v.mean_ms.toFixed(3)+\' | 중앙값 \'+v.median_ms.toFixed(3)+\' | p95 \'+v.p95_ms.toFixed(3)+\' ms\');\n  if(e.speedup)lines.push(\'CPU/GPU 총 계산시간 비율: \'+e.speedup.toFixed(2)+\'×\');\n  $(\'batchSummary\').textContent=lines.join(\' / \');lines.forEach(batchLine);\n  batchLine(\'서버 경과시간 \'+e.batch_wall_ms.toFixed(1)+\' ms (로그·검증 포함). 위 총시간은 프레임 렌더링 시간의 합.\');\n }\n if(e.type===\'error\')throw Error(e.message);\n}\n$(\'batchRun\').onclick=async()=>{\n const k=Number($(\'batchK\').value);if(!Number.isInteger(k)||k<1||k>300){$(\'batchSummary\').textContent=\'K는 1~300의 정수입니다.\';return;}\n running=false;$(\'play\').textContent=\'경로 재생\';batching=true;\n while(busy)await new Promise(r=>setTimeout(r,20));pending=false;\n const p=new URLSearchParams();[\'model\',\'side\',\'cap\',\'path\',\'yaw\',\'pitch\',\'distance\',\'phase\'].forEach(x=>p.set(x,$(x).value));p.set(\'k\',k);p.set(\'mode\',$(\'batchMode\').value);p.set(\'upload\',$(\'upload\').checked?\'1\':\'0\');\n const controls=[...document.querySelectorAll(\'input,select,button\')].filter(e=>![\'batchStop\',\'batchSave\'].includes(e.id));\n controls.forEach(e=>e.disabled=true);$(\'batchStop\').disabled=false;$(\'batchSave\').disabled=true;\n $(\'batchLog\').textContent=\'\';$(\'batchSummary\').textContent=\'장면 준비·warmup 중…\';batchRows=[];batchAbort=new AbortController();\n try{\n  const r=await fetch(\'benchmark?\'+p,{signal:batchAbort.signal});if(!r.ok)throw Error(await r.text());\n  const reader=r.body.getReader(),decoder=new TextDecoder();let buf=\'\';\n  while(true){const {done,value}=await reader.read();buf+=decoder.decode(value||new Uint8Array(),{stream:!done});let lines=buf.split(\'\\n\');buf=lines.pop();for(const line of lines)if(line.trim())batchEvent(JSON.parse(line));if(done){if(buf.trim())batchEvent(JSON.parse(buf));break;}}\n }catch(e){batchLine(e.name===\'AbortError\'?\'사용자가 중단했습니다.\':\'오류: \'+e.message);$(\'batchSummary\').textContent=e.name===\'AbortError\'?\'중단됨\':\'측정 오류\';}\n finally{batching=false;controls.forEach(e=>e.disabled=false);$(\'batchStop\').disabled=true;$(\'batchSave\').disabled=!batchRows.length;batchAbort=null;}\n};\n$(\'batchStop\').onclick=()=>batchAbort?.abort();\n$(\'batchSave\').onclick=()=>{let url=URL.createObjectURL(new Blob([JSON.stringify(batchRows,null,2)],{type:\'application/json\'}));let a=document.createElement(\'a\');a.href=url;a.download=\'rt616-k-frames.json\';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);};\nframe();</script>"""\nclass Handler(BaseHTTPRequestHandler):\n    def setup(self):super().setup();self.connection.setsockopt(socket.IPPROTO_TCP,socket.TCP_NODELAY,1)\n    def do_GET(self):\n        p=urlparse(self.path)\n        if p.path==\'/\':\n            b=HTML.replace(\'__GPU_AVAILABLE__\',\'true\' if GPU_AVAILABLE else \'false\').encode();self.send_response(200);self.send_header(\'Content-Type\',\'text/html; charset=utf-8\');self.end_headers();self.wfile.write(b);return\n        if p.path==\'/benchmark\':self.benchmark(p);return\n        if p.path!=\'/frame\':self.send_error(404);return\n        try:\n            q=parse_qs(p.query);get=lambda k,d:q.get(k,[d])[0]\n            model=get(\'model\',\'Bunny\');side=int(get(\'side\',\'8\'));cap=int(get(\'cap\',\'36000\'));path=get(\'path\',\'Hover\')\n            if model not in MODELS or cap not in [12000,36000,100000] or path not in PATHS:raise ValueError(\'Invalid selection\')\n            s=scene(model,side,cap);R,e=pose(float(get(\'phase\',\'0\')),s.extent,path,float(get(\'yaw\',\'0\')),float(get(\'pitch\',\'25\')),float(get(\'distance\',\'1.2\')))\n            start=time.perf_counter();d=s.frame(R,e,get(\'backend\',\'GPU\'),get(\'upload\',\'0\')==\'1\');ms=(time.perf_counter()-start)*1000\n            rgb=depth_rgb(d,far=s.extent*3);rgba=np.full((480,640,4),255,np.uint8);rgba[:,:,:3]=rgb;b=rgba.tobytes()\n            ctype=\'application/octet-stream\'\n            if get(\'format\',\'raw\')==\'png\':\n                from PIL import Image\n                stream=io.BytesIO();Image.fromarray(rgb).save(stream,format=\'PNG\',compress_level=1);b=stream.getvalue();ctype=\'image/png\'\n            self.send_response(200);self.send_header(\'Content-Type\',ctype);self.send_header(\'Cache-Control\',\'no-store\');self.send_header(\'Content-Length\',str(len(b)))\n            for k,v in {\'X-Render-Ms\':f\'{ms:.3f}\',\'X-Points\':str(s.n),\'X-Stored-MiB\':f\'{s.info["stored_bytes"]/2**20:.2f}\',\'X-Eye\':\',\'.join(f\'{x:.2f}\' for x in e)}.items():self.send_header(k,v)\n            self.end_headers();self.wfile.write(b)\n        except Exception as ex:self.send_error(400,str(ex))\n\n    def benchmark(self,p):\n        started=False\n        try:\n            q=parse_qs(p.query);get=lambda k,d:q.get(k,[d])[0]\n            model=get(\'model\',\'Bunny\');cap=int(get(\'cap\',\'36000\'));path=get(\'path\',\'Hover\')\n            if model not in MODELS or cap not in [12000,36000,100000] or path not in PATHS:raise ValueError(\'Invalid selection\')\n            s=scene(model,int(get(\'side\',\'8\')),cap)\n            events=benchmark_frames(s,int(get(\'k\',\'60\')),path,float(get(\'phase\',\'0\')),float(get(\'yaw\',\'0\')),float(get(\'pitch\',\'25\')),float(get(\'distance\',\'1.2\')),get(\'mode\',\'Both\'),get(\'upload\',\'0\')==\'1\')\n            first=next(events)\n            self.send_response(200);self.send_header(\'Content-Type\',\'application/x-ndjson; charset=utf-8\');self.send_header(\'Cache-Control\',\'no-store\');self.send_header(\'X-Accel-Buffering\',\'no\');self.end_headers();started=True\n            def send(event):self.wfile.write((json.dumps(event)+\'\\n\').encode());self.wfile.flush()\n            send(first)\n            for event in events:send(event)\n        except (BrokenPipeError,ConnectionResetError,ConnectionAbortedError):pass\n        except Exception as ex:\n            if not started:self.send_error(400,str(ex))\n            else:\n                try:self.wfile.write((json.dumps(dict(type=\'error\',message=str(ex)))+\'\\n\').encode());self.wfile.flush()\n                except OSError:pass\n\n    def log_message(self,*args):pass\nif __name__==\'__main__\':\n    import os\n    import argparse\n    parser=argparse.ArgumentParser(description=\'Stanford point-cloud CPU/CUDA viewer\')\n    parser.add_argument(\'--port\',type=int,default=int(os.environ.get(\'RT616_DEMO_PORT\',\'8766\')))\n    args=parser.parse_args();port=args.port\n    print(\'Preparing Bunny (first run downloads the Stanford archive)...\',flush=True)\n    scene(\'Bunny\',8,36000);print(f\'Open http://127.0.0.1:{port}\',flush=True);HTTPServer((\'127.0.0.1\',port),Handler).serve_forever()\n',encoding='utf-8')
Path('colab_live.py').write_text('"""Launch the actual CPU/CUDA depth server inside a Colab output frame."""\nimport os,sys,subprocess,time,urllib.request\nfrom pathlib import Path\n_process=None\ndef launch(port=8766,show=True):\n    global _process\n    if _process is not None and _process.poll() is None:\n        if show:display_live(port)\n        return _process\n    # Never terminate a process that this launcher did not create.\n    import socket\n    with socket.socket() as sock:\n        if sock.connect_ex((\'127.0.0.1\',port))==0:\n            raise RuntimeError(f\'Port {port} is already in use. Use launch(port={port+1}).\')\n    folder=Path(__file__).resolve().parent\n    env=os.environ.copy();env[\'RT616_DEMO_PORT\']=str(port)\n    with (folder/\'live_server.log\').open(\'w\',encoding=\'utf-8\') as log:\n        _process=subprocess.Popen([sys.executable,\'-u\',str(folder/\'serve_demo.py\')],cwd=folder,env=env,stdout=log,stderr=subprocess.STDOUT)\n    for _ in range(240):\n        if _process.poll() is not None:raise RuntimeError((folder/\'live_server.log\').read_text(encoding=\'utf-8\'))\n        try:\n            with urllib.request.urlopen(f\'http://127.0.0.1:{port}/\',timeout=1) as r:\n                if r.status==200:break\n        except Exception:time.sleep(.5)\n    else:\n        stop();raise RuntimeError(\'Server startup timed out; inspect live_server.log and rerun.\')\n    if show:display_live(port)\n    return _process\ndef display_live(port=8766):\n    try:\n        from google.colab import output\n    except ImportError:\n        from IPython.display import IFrame,display\n        display(IFrame(f\'http://127.0.0.1:{port}/?transport=png\',width=\'100%\',height=1000))\n    else:\n        output.serve_kernel_port_as_iframe(port,path=\'/?transport=png\',height=1000,cache_in_notebook=False)\n    print(\'화면의 경로 재생을 누르세요. 드래그=각도, 휠=접근/후퇴. 정지 버튼으로 일시정지.\')\ndef stop():\n    global _process\n    if _process is not None and _process.poll() is None:\n        _process.terminate()\n        try:_process.wait(timeout=10)\n        except subprocess.TimeoutExpired:_process.kill();_process.wait()\n    _process=None\n\n',encoding='utf-8')
Path('benchmark_lab.py').write_text('"""Fixed-pose batches: rendering timings exclude image encoding and transport."""\nfrom stanford_scene import *\ndef benchmark_frames(scene,k=60,path=\'Hover\',phase=0,yaw=0,pitch=25,distance=1.2,mode=\'Both\',upload=False):\n    if not 1<=k<=300:raise ValueError(\'K must be between 1 and 300\')\n    if mode not in [\'Both\',\'CPU\',\'GPU\']:raise ValueError(\'Invalid backend\')\n    backends=[\'CPU\',\'GPU\'] if mode==\'Both\' else [mode]\n    if \'GPU\' in backends and not GPU_AVAILABLE:raise RuntimeError(\'GPU unavailable; select CPU or enable GPU runtime\')\n    phases=[(phase+i/k)%1 for i in range(k)]\n    poses=[pose(t,scene.extent,path,yaw,pitch,distance) for t in phases]\n    environment=dict(python=platform.python_version(),os=platform.system(),numpy=np.__version__,gpu_available=GPU_AVAILABLE)\n    if GPU_AVAILABLE:\n        props=cp.cuda.runtime.getDeviceProperties(0);name=props[\'name\']\n        environment.update(gpu=name.decode() if isinstance(name,bytes) else name,cupy=cp.__version__,cuda_runtime=cp.cuda.runtime.runtimeGetVersion())\n    yield dict(type=\'start\',environment=environment,resolution=[640,480],dtype=\'float32\',k=k,scene=scene.info,path=path,backends=backends,upload=upload,\n               measurement=\'render+GPU pose upload+depth readback; image encoding/transfer/display excluded\')\n    # Compile and stabilize before recording. Results must finish on the host.\n    for backend in backends:\n        for _ in range(3):scene.frame(*poses[0],backend,upload)\n    yield dict(type=\'warmup\',message=\'Warmup complete: 3 frames/backend, excluded.\')\n    totals={b:[] for b in backends};start_batch=time.perf_counter()\n    for i,(R,e) in enumerate(poses):\n        row=dict(type=\'frame\',frame=i+1,k=k,phase=phases[i],ms={})\n        results={}\n        # Alternate execution order to reduce systematic first/second bias.\n        order=backends if i%2==0 else list(reversed(backends))\n        for backend in order:\n            start=time.perf_counter();depth=scene.frame(R,e,backend,upload);ms=(time.perf_counter()-start)*1000\n            row[\'ms\'][backend]=ms;totals[backend].append(ms)\n            if i in [0,k-1]:results[backend]=depth\n        if len(results)==2:\n            a,b=results[\'CPU\'],results[\'GPU\'];mask=np.isfinite(a)\n            row[\'matching\']=bool(np.array_equal(mask,np.isfinite(b)) and np.allclose(a[mask],b[mask],rtol=1e-5,atol=1e-5))\n        print(f"[{i+1:03d}/{k}] "+\' | \'.join(f\'{b}: {row["ms"][b]:.3f} ms\' for b in backends),flush=True)\n        yield row\n    stats={b:dict(total_ms=float(np.sum(v)),mean_ms=float(np.mean(v)),median_ms=float(np.median(v)),\n                  p95_ms=float(np.percentile(v,95))) for b,v in totals.items()}\n    yield dict(type=\'done\',stats=stats,batch_wall_ms=(time.perf_counter()-start_batch)*1000,\n               speedup=stats[\'CPU\'][\'total_ms\']/stats[\'GPU\'][\'total_ms\'] if len(backends)==2 else None)\n\n',encoding='utf-8')
print('준비 완료. ② 뷰어 셀을 실행하세요.')

In [ ]:
#@title ② 라이브 뷰어 + K 프레임 비교
import colab_live, importlib
colab_live.stop()  # 이 노트북이 띄운 이전 서버만 종료
importlib.reload(colab_live)
live_process = colab_live.launch()


## 선택: 뷰어 없이 셀에 프레임별 시간 출력
아래 **③**은 같은 K 프레임 실험을 텍스트 로그로만 실행합니다. 뷰어의 자동 재생을 정지한 뒤 실행하세요.
K·GRID·MODEL·CAMERA_PATH를 바꿀 수 있습니다. 최초 준비·3회 warmup은 측정에서 제외됩니다.


In [ ]:
#@title ③ 선택 — 콘솔에서 K 프레임 비교
from stanford_scene import Scene
from benchmark_lab import benchmark_frames
K, GRID, MODEL, CAMERA_PATH = 10, 8, 'Bunny', 'Hover'
scene = Scene(MODEL, GRID, 36000)
results = list(benchmark_frames(scene, k=K, path=CAMERA_PATH, mode='Both'))
for backend, stats in results[-1]['stats'].items():
    print(backend, {key:round(value,3) for key,value in stats.items()})
import json
Path('k_frames_result.json').write_text(json.dumps(results,indent=2))
print('Saved k_frames_result.json')


**해석:** CPU는 Numba 단일 스레드, GPU는 CUDA입니다. 모델당 샘플 상한과 실제 처리 점 수를 화면에서 확인하세요. 점과 복제 위치를 따로 저장하지만 모든 복제 점을 투영합니다.
실제 프레임 처리 시간은 장치·장면·카메라 위치에 따라 달라집니다.
화면 재표시: colab_live.display_live() / 서버 종료: colab_live.stop().

출처: [Stanford 3D Scanning Repository](https://graphics.stanford.edu/data/3Dscanrep/) · [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/index.html)
